# Projet — Transfer Learning ResNet18 sur UrbanSound8K

Ce notebook applique le **transfer learning** avec **ResNet18** pré-entraîné sur ImageNet pour classifier les 10 classes de sons urbains du dataset UrbanSound8K.

On compare cette approche au CNN custom entraîné depuis zéro dans `entrainement_urban_sound_8k.ipynb`.

**Principe du transfer learning :**
- On part d'un ResNet18 qui a déjà appris à extraire des caractéristiques visuelles pertinentes sur 1.2 million d'images ImageNet.
- On gèle le backbone (toutes les couches convolutives) pour conserver ces connaissances.
- On remplace uniquement la dernière couche linéaire pour adapter le modèle à nos 10 classes.
- On entraîne seulement cette nouvelle tête — beaucoup moins de paramètres, convergence plus rapide.

## Imports

In [1]:
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.models as models
import torchvision.transforms as T
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Chargement des métadonnées

Même dataset et même découpage que dans `entrainement_urban_sound_8k.ipynb` :
- 10 classes urbaines, 8 732 échantillons au total
- Folds 1–9 en entraînement (~7 859 échantillons), fold 10 en test (~873 échantillons)

In [2]:
META_PATH = Path("metadata/UrbanSound8K.csv")
AUDIO_DIR = Path("audio")

df = pd.read_csv(META_PATH)

print(f"Échantillons total : {len(df)}")
print(f"Nombre de classes  : {df['class'].nunique()}")
print(f"Classes            : {sorted(df['class'].unique())}")
df.head()

Échantillons total : 8732
Nombre de classes  : 10
Classes            : ['air_conditioner', 'car_horn', 'children_playing', 'dog_bark', 'drilling', 'engine_idling', 'gun_shot', 'jackhammer', 'siren', 'street_music']


,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


## Extraction du spectrogramme Mel

Même pipeline que dans le notebook CNN — les spectrogrammes ont la forme `(128, 173)` après zero-padding à 4 secondes.

Pour ResNet18, on devra ensuite :
1. **Répliquer le canal** : passer de `(1, 128, 173)` à `(3, 128, 173)` pour simuler une image RGB.
2. **Redimensionner** : passer à `(3, 224, 224)` — la taille attendue par ResNet18.
3. **Normaliser** avec les statistiques ImageNet.

> **Pourquoi les stats ImageNet ?** Le backbone ResNet18 a été entraîné avec ces valeurs. Utiliser la même normalisation assure que les activations de ses couches restent dans les plages attendues, même si l'entrée n'est pas une vraie photo.

In [3]:
SR         = 22050
DURATION   = 4.0
N_MELS     = 128
HOP_LENGTH = 512


def load_spectrogram(fold, filename):
    """Charge un fichier WAV et retourne son spectrogramme Mel en dB (128 × 173)."""
    path = AUDIO_DIR / f"fold{fold}" / filename
    signal, _ = librosa.load(path, sr=SR, duration=DURATION)

    target_len = int(SR * DURATION)
    if len(signal) < target_len:
        signal = librosa.util.fix_length(signal, size=target_len)

    mel = librosa.feature.melspectrogram(
        y=signal, sr=SR, n_mels=N_MELS, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=mel.max())
    return mel_db


sample = df.iloc[0]
spec = load_spectrogram(sample["fold"], sample["slice_file_name"])
print(f"Forme du spectrogramme brut : {spec.shape}  (n_mels × trames temporelles)")
print(f"Classe                      : {sample['class']}")

d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Forme du spectrogramme brut : (128, 173)  (n_mels × trames temporelles)
Classe                      : dog_bark


## Dataset PyTorch — adaptation pour ResNet18

ResNet18 attend des images de taille `(3, 224, 224)`. Nos spectrogrammes sont `(1, 128, 173)`. On adapte dans le `__getitem__` :

1. `unsqueeze(0)` → `(1, 128, 173)`
2. `.repeat(3, 1, 1)` → `(3, 128, 173)` : on copie le canal 3 fois pour simuler RGB
3. `T.Resize((224, 224))` → `(3, 224, 224)` : interpolation bilinéaire
4. `T.Normalize(ImageNet stats)` : normalisation avec la moyenne et l'écart-type d'ImageNet

Le mapping classe→indice utilise directement les `classID` du CSV (0–9), cohérents avec les labels officiels.

In [4]:
resnet_transform = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class UrbanSoundResNetDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = load_spectrogram(row["fold"], row["slice_file_name"])   # (128, 173)
        x = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)       # (1, 128, 173)
        x = x.repeat(3, 1, 1)                                           # (3, 128, 173)
        x = resnet_transform(x)                                          # (3, 224, 224)
        y = int(row["classID"])
        return x, y


df_train = df[df["fold"] != 10]
df_test  = df[df["fold"] == 10]

train_dataset = UrbanSoundResNetDataset(df_train)
test_dataset  = UrbanSoundResNetDataset(df_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print(f"Train : {len(train_dataset)} échantillons  |  Test : {len(test_dataset)} échantillons")
print(f"Forme d'un batch : {x_batch.shape}  →  labels : {y_batch.shape}")

Train : 7895 échantillons  |  Test : 837 échantillons
Forme d'un batch : torch.Size([64, 3, 224, 224])  →  labels : torch.Size([64])


## Construction du modèle — ResNet18 fine-tuné

**Stratégie :**
1. Charger ResNet18 avec les poids ImageNet pré-entraînés.
2. **Geler** tous les paramètres du backbone (`requires_grad = False`) — ils ne seront pas mis à jour.
3. **Remplacer** la couche finale `fc` (512 → 1000 classes ImageNet) par une nouvelle couche (512 → 10 classes urbaines).
4. Seule cette nouvelle tête est entraînable.

```
ResNet18 (backbone gelé)
  → conv1, layer1, layer2, layer3, layer4  [poids ImageNet fixés]
  → avgpool                                 [poids ImageNet fixés]
  → fc : Linear(512 → 10)                  [seule partie entraînable]
```

Avantage : au lieu d'entraîner **11.2M paramètres** depuis zéro, on n'entraîne que **5 130 paramètres** (512×10 + 10 biais).

In [5]:
NUM_CLASSES = 10

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Linear(512, NUM_CLASSES)

resnet = resnet.to(device)

n_trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in resnet.parameters())
print(f"Paramètres total        : {n_total:,}")
print(f"Paramètres entraînables : {n_trainable:,}  (tête uniquement)")

x_dummy = torch.randn(1, 3, 224, 224).to(device)
out = resnet(x_dummy)
print(f"Forward pass : entrée {tuple(x_dummy.shape)} → sortie {tuple(out.shape)}")

Paramètres total        : 11,181,642
Paramètres entraînables : 5,130  (tête uniquement)
Forward pass : entrée (1, 3, 224, 224) → sortie (1, 10)


## Entraînement

Même configuration que le CNN custom pour une comparaison équitable :
- **`CrossEntropyLoss`** : perte standard pour classification multi-classes
- **`AdamW`** (lr=1e-3) : on optimise uniquement `resnet.fc.parameters()`
- **`ReduceLROnPlateau`** : réduit le LR si la validation stagne (patience=3, factor=0.5)
- **Early stopping** (patience=5) : arrête si pas d'amélioration, restaure les meilleurs poids
- **TensorBoard** : logs dans `runs/resnet18_urban_sound_8k/`

In [6]:
def train_loop(dataloader, model, loss_fn, optimizer, writer, epoch):
    model.train()
    pbar = tqdm(dataloader, desc="Train")
    for batch, (x, y_true) in enumerate(pbar):
        x, y_true = x.to(device), y_true.to(device)
        y_pred = model(x)
        loss = loss_fn(y_pred, y_true)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item():.4f}")
        writer.add_scalar("Loss/train", loss.item(), epoch * len(dataloader) + batch)


def test_loop(dataloader, model, loss_fn, optimizer, writer, epoch):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for x, y_true in tqdm(dataloader, desc="Test "):
            x, y_true = x.to(device), y_true.to(device)
            y_pred = model(x)
            test_loss += loss_fn(y_pred, y_true).item()
            correct += (y_pred.argmax(1) == y_true).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Accuracy : {100 * correct:.1f}%  |  Avg loss : {test_loss:.6f}")

    writer.add_scalar("Loss/test", test_loss, epoch)
    writer.add_scalar("Accuracy/test", 100 * correct, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    return test_loss, correct

In [7]:
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.AdamW(resnet.fc.parameters(), lr=1e-3)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
writer     = SummaryWriter("runs/resnet18_urban_sound_8k")

In [8]:
epochs      = 10
es_patience = 5

best_loss    = float("inf")
best_acc     = 0.0
best_weights = None
epochs_without_improvement = 0

for t in range(epochs):
    print(f"Epoch : {t+1}  |  lr = {optimizer.param_groups[0]['lr']:.2e}")
    train_loop(train_loader, resnet, criterion, optimizer, writer, t)
    val_loss, val_acc = test_loop(test_loader, resnet, criterion, optimizer, writer, t)

    scheduler.step(val_loss)

    if val_loss < best_loss:
        best_loss    = val_loss
        best_acc     = val_acc
        best_weights = {k: v.cpu().clone() for k, v in resnet.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        print(f"  (pas d'amélioration depuis {epochs_without_improvement} époque(s))")
        if epochs_without_improvement >= es_patience:
            print(f"Early stopping déclenché à l'époque {t+1}.")
            break

resnet.load_state_dict(best_weights)
print(f"\nEntraînement terminé. Meilleure précision : {100 * best_acc:.1f}%")

Epoch : 1  |  lr = 1.00e-03


Test : 100%|██████████| 14/14 [00:06<00:00,  2.30it/s]


Accuracy : 63.1%  |  Avg loss : 1.056590
Epoch : 2  |  lr = 1.00e-03


Test : 100%|██████████| 14/14 [00:06<00:00,  2.30it/s]


Accuracy : 70.3%  |  Avg loss : 0.863748
Epoch : 3  |  lr = 1.00e-03


...

Accuracy : 72.3%  |  Avg loss : 0.801662
  (pas d'amélioration depuis 2 époque(s))
Epoch : 9  |  lr = 1.00e-03


Test : 100%|██████████| 14/14 [00:06<00:00,  2.22it/s]


Accuracy : 73.0%  |  Avg loss : 0.787530
  (pas d'amélioration depuis 3 époque(s))
Epoch : 10  |  lr = 1.00e-03


Test : 100%|██████████| 14/14 [00:06<00:00,  2.18it/s]

Accuracy : 72.9%  |  Avg loss : 0.819966
  (pas d'amélioration depuis 4 époque(s))

Entraînement terminé. Meilleure précision : 71.6%


## Évaluation finale

On calcule la **matrice de confusion** et les **métriques par classe** (precision, recall, F1) sur l'ensemble de test pour identifier les classes bien reconnues et celles qui posent problème.

In [9]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

CLASS_NAMES = [
    "air_conditioner", "car_horn", "children_playing", "dog_bark",
    "drilling", "engine_idling", "gun_shot", "jackhammer",
    "siren", "street_music"
]

resnet.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y_true in test_loader:
        x = x.to(device)
        y_pred = resnet(x).argmax(1).cpu()
        all_preds.extend(y_pred.tolist())
        all_labels.extend(y_true.tolist())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
)
ax.set_xlabel("Prédit")
ax.set_ylabel("Réel")
ax.set_title("Matrice de confusion — ResNet18 UrbanSound8K (fold 10)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
writer.add_figure("Confusion Matrix", fig)
plt.show()

writer.close()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

                  precision    recall  f1-score   support

 air_conditioner       0.68      0.73      0.70       100
        car_horn       0.83      0.45      0.59        33
children_playing       0.62      0.63      0.63       100
        dog_bark       0.69      0.70      0.70       100
        drilling       0.73      0.61      0.66       100
   engine_idling       0.81      0.70      0.75        93
        gun_shot       0.91      0.94      0.92        32
      jackhammer       0.74      0.94      0.83        96
           siren       0.78      0.54      0.64        83
    street_music       0.66      0.87      0.75       100

        accuracy                           0.72       837
       macro avg       0.74      0.71      0.72       837
    weighted avg       0.72      0.72      0.71       837



## Export ONNX

**ONNX** (Open Neural Network Exchange) est un format ouvert qui permet de déployer un modèle PyTorch dans n'importe quel environnement compatible : moteurs d'inférence haute performance (ONNX Runtime, TensorRT), applications mobiles, navigateurs (via ONNX.js), etc.

L'export fonctionne par **traçage** : PyTorch effectue un forward pass avec un tenseur factice (*dummy input*) et enregistre toutes les opérations dans un graphe statique.

Points clés :
- Le modèle doit être en mode **`eval()`** avant l'export pour désactiver Dropout et BatchNorm en mode entraînement.
- `input_names` et `output_names` donnent des noms lisibles aux entrées/sorties du graphe.
- `dynamic_axes` permet d'accepter des batches de taille variable à l'inférence (`batch_size` n'est pas fixé dans le graphe).
- `opset_version=17` cible une version ONNX récente et largement supportée.

In [10]:
import torch
from pathlib import Path

ONNX_DIR  = Path("onnx")
ONNX_DIR.mkdir(exist_ok=True)
ONNX_PATH = ONNX_DIR / "resnet18_urban_sound_8k.onnx"

resnet.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    resnet,
    dummy_input,
    str(ONNX_PATH),
    opset_version=17,
    input_names=["spectrogram"],
    output_names=["logits"],
    dynamic_axes={
        "spectrogram": {0: "batch_size"},
        "logits":      {0: "batch_size"},
    },
)

print(f"Modèle exporté : {ONNX_PATH}")

C:\Users\arnau\AppData\Local\Temp\ipykernel_23624\709147884.py:12: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0611 13:54:50.736000 23624 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0611 13:54:51.391000 23624 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\arnau\AppData\Local\Python\pythoncore-3.14-64\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "d:\Projets\Code\IIM\cours_ia_2\.venv\Lib\site

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Modèle exporté : onnx\resnet18_urban_sound_8k.onnx


## Comparaison des résultats

| Modèle | Paramètres entraînables | Précision test (fold 10) |
|--------|------------------------|---------------------------|
| CNN custom (depuis zéro) | ~11M | à compléter |
| ResNet18 (tête seule) | 5 130 | à compléter |

**Observations attendues :**
- Le ResNet18 fine-tuné converge beaucoup plus vite (moins d'époques nécessaires).
- Avec seulement 5 130 paramètres à entraîner, le risque de sur-apprentissage est très réduit.
- La précision devrait être supérieure grâce aux représentations apprises sur ImageNet, qui captent des structures locales réutilisables même pour des spectrogrammes.